1. Data Processing & Preparation
We begin by reading the medical_insurance.csv file, encoding categorical variables, and standardizing features (scaling is absolutely critical for PCR and PLSR).

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import statsmodels.api as sm

# Load the dataset
df = pd.read_csv("medical_insurance.csv")

# Define target and potential predictors
target = "annual_medical_cost"
continuous_features = ["age", "income", "bmi", "visits_last_year", "risk_score", "claims_count"]
categorical_features = ["sex", "region", "urban_rural", "smoker", "plan_type"]

# Clean missing records
df_clean = df.dropna(subset=continuous_features + categorical_features + [target])

X_raw = df_clean[continuous_features + categorical_features]
y = df_clean[target]

# One-hot encoding
X_encoded = pd.get_dummies(X_raw, columns=categorical_features, drop_first=True).astype(float)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# Scale data (Mandatory for PCR/PLSR dimensional stability)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_encoded.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_encoded.columns)

2. Forward and Backward SelectionStepwise selection adds or removes features one-by-one based on statistical significance ($p$-values).

In [3]:
def forward_selection(X, y, significance_level=0.05):
    """Select features by iteratively adding the most statistically significant variable."""
    initial_features = X.columns.tolist()
    best_features = []
    while len(initial_features) > 0:
        remaining_features = list(set(initial_features) - set(best_features))
        new_pval = pd.Series(index=remaining_features, dtype=float)
        for new_column in remaining_features:
            model = sm.OLS(y, sm.add_constant(X[best_features + [new_column]])).fit()
            new_pval[new_column] = model.pvalues[new_column]
        min_p_value = new_pval.min()
        if min_p_value < significance_level:
            best_features.append(new_pval.idxmin())
        else:
            break
    return best_features

def backward_elimination(X, y, significance_level=0.05):
    """Select features by iteratively removing the least statistically significant variable."""
    features = X.columns.tolist()
    while len(features) > 0:
        model = sm.OLS(y, sm.add_constant(X[features])).fit()
        p_values = model.pvalues.drop('const')
        max_p_value = p_values.max()
        if max_p_value > significance_level:
            excluded_feature = p_values.idxmax()
            features.remove(excluded_feature)
        else:
            break
    return features

# Run both algorithms
forward_feats = forward_selection(X_train_scaled, y_train.values)
backward_feats = backward_elimination(X_train_scaled, y_train.values)

print("--- Stepwise Feature Selection Results ---")
print(f"Forward Selection Chosen Features ({len(forward_feats)}): {forward_feats}\n")
print(f"Backward Elimination Chosen Features ({len(backward_feats)}): {backward_feats}")

--- Stepwise Feature Selection Results ---
Forward Selection Chosen Features (6): ['claims_count', 'risk_score', 'age', 'smoker_Never', 'visits_last_year', 'region_South']

Backward Elimination Chosen Features (6): ['age', 'visits_last_year', 'risk_score', 'claims_count', 'region_South', 'smoker_Never']


3. Principal Component Regression (PCR)
PCR applies Principal Component Analysis (PCA) on your features to compress them into orthogonal dimensions, then passes those dimensions to an Ordinary Least Squares (OLS) Linear Regression model.

In [4]:
# Step 1: Initialize PCA and determine how many components to retain
# Let's reduce our total dimensions down to 5 distinct unsupervised components
n_components = 5
pca = PCA(n_components=n_components)

# Step 2: Fit and transform scaled features into components
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Step 3: Fit OLS Regression on top of the components
pcr_model = LinearRegression()
pcr_model.fit(X_train_pca, y_train)

# Evaluate PCR
pcr_preds = pcr_model.predict(X_test_pca)
print("--- Principal Component Regression (PCR) ---")
print(f"Explained Variance Ratio by Component: {pca.explained_variance_ratio_}")
print(f"PCR Test R² Score: {r2_score(y_test, pcr_preds):.4f}")
print(f"PCR Test RMSE: {np.sqrt(mean_squared_error(y_test, pcr_preds)):.2f}")

--- Principal Component Regression (PCR) ---
Explained Variance Ratio by Component: [0.11241295 0.09032142 0.08983966 0.08089131 0.07333958]
PCR Test R² Score: 0.0830
PCR Test RMSE: 3003.92


4. Partial Least Squares Regression (PLSR)
Unlike PCR, PLSR dynamically inspects the target labels (y_train) while generating components to ensure that the dimensional reductions map directly to predictive output directions.

In [5]:
# Initialize PLSR with the same number of target components (e.g., 5)
plsr_model = PLSRegression(n_components=5)

# Train the model
plsr_model.fit(X_train_scaled, y_train)

# Evaluate PLSR
# Note: scikit-learn's PLSR predict output shape can be 2D, so we flatten it using .ravel()
plsr_preds = plsr_model.predict(X_test_scaled).ravel()

print("\n--- Partial Least Squares Regression (PLSR) ---")
print(f"PLSR Test R² Score: {r2_score(y_test, plsr_preds):.4f}")
print(f"PLSR Test RMSE: {np.sqrt(mean_squared_error(y_test, plsr_preds)):.2f}")


--- Partial Least Squares Regression (PLSR) ---
PLSR Test R² Score: 0.1163
PLSR Test RMSE: 2948.82


5. Optimal Modeling Selection Engine
To select the correct pipeline based on the dataset's unique attributes, we can construct an automated evaluation function. It assesses whether low-dimensional compression (PCR/PLSR) or pure feature pruning (Stepwise) produces the highest generalization capability (lowest Root Mean Squared Error).

In [6]:
def design_optimal_dimension_strategy(X_tr, X_te, y_tr, y_te, fwd_features, bwd_features, pcr_p, plsr_p):
    """
    Evaluates and selects the best modeling strategy among Stepwise Selection,
    PCR, or PLSR based on performance optimization.
    """
    results = {}
    
    # 1. Evaluate Forward Selection Model
    if fwd_features:
        m = LinearRegression().fit(X_tr[fwd_features], y_tr)
        results["Forward Selection"] = np.sqrt(mean_squared_error(y_te, m.predict(X_te[fwd_features])))
        
    # 2. Evaluate Backward Selection Model
    if bwd_features:
        m = LinearRegression().fit(X_tr[bwd_features], y_tr)
        results["Backward Elimination"] = np.sqrt(mean_squared_error(y_te, m.predict(X_te[bwd_features])))
        
    # 3. Evaluate PCR & PLSR Metrics
    results["Principal Component Regression (PCR)"] = np.sqrt(mean_squared_error(y_te, pcr_p))
    results["Partial Least Squares Regression (PLSR)"] = np.sqrt(mean_squared_error(y_te, plsr_p))
    
    # Sort strategies to discover the absolute lowest error rate
    best_strategy = min(results, key=results.get)
    
    print("\n================ FINAL STRATEGIC DESIGN SELECTION ================")
    for strategy, rmse_val in results.items():
        print(f"• {strategy} - Test RMSE: {rmse_val:.2f}")
        
    print(f"\n🚀 RECOMMENDATION: Adopt the **{best_strategy}** method.")
    
    # Structural Context Logic Breakdowns
    if "Selection" in best_strategy or "Elimination" in best_strategy:
        print("REASONING: The dataset features perform best when uncompressed. Keeping distinct columns preserves full interpretation metrics.")
    elif "PLSR" in best_strategy:
        print("REASONING: Strong underlying multi-collinearity or directional variance exists. Supervised target projection yields optimal feature alignment.")
    else:
        print("REASONING: Unsupervised variances dominate feature space. Isolating raw components before regression protects prediction weights.")

# Run Strategy Analysis
design_optimal_dimension_strategy(
    X_train_scaled, X_test_scaled, y_train, y_test, 
    forward_feats, backward_feats, pcr_preds, plsr_preds
)


================ FINAL STRATEGIC DESIGN SELECTION ================
• Forward Selection - Test RMSE: 2948.78
• Backward Elimination - Test RMSE: 2948.78
• Principal Component Regression (PCR) - Test RMSE: 3003.92
• Partial Least Squares Regression (PLSR) - Test RMSE: 2948.82

🚀 RECOMMENDATION: Adopt the **Forward Selection** method.
REASONING: The dataset features perform best when uncompressed. Keeping distinct columns preserves full interpretation metrics.
